# Hard-disagreement experiment — native-v3 → native-v4

Warm-start your completed **native-v3 rank-16 LoRA and pointer head**. V4 uses its own output, recovery and fresh optimizer/schedule; v1/v2/v3 stay available. No general-decision curriculum rerun is needed.

The [v3 audit](https://github.com/yxc20089/QPlusLearning/blob/main/labs/lab-01-kev-pacman/evaluation/v3-analysis-2026-10-07/README.md) found fewer immediate avoidable deaths, but zero clears and worse food efficiency. This experiment changes data selection: mostly verified consequential learner/teacher disagreements, with a bounded replay budget. Rank, architecture, observation/action protocol, CE loss and total update budget stay fixed. Read [the experiment plan](https://github.com/yxc20089/QPlusLearning/blob/main/labs/lab-01-kev-pacman/evaluation/v4-hard-disagreements-plan.md) before generation.

**Generation is prework, not a 30-minute class activity.** Offline native scenarios and teacher continuation verification use concurrent CPU workers; v3 collection/probes and training use the optimized GPU runtime. No v4 improvement or generation duration is claimed before an actual run. Hard quotas must pass; the code does not fill missing cases with easy frames. The main lab still has one assessed CP1.

For a restored completed v4 checkpoint, run setup/storage and the adapter-loading cell, then skip generation/training and go to comparison or interactive play.

## Setup and Drive storage

Use the same RTX PRO 6000 Blackwell allocation and pinned optimized training environment. Drive is enabled by default. Storage restores available v3/v4 checkpoints and v4's exact dataset/receipt/development/replay evidence. The adapter-loading cell separately restores completed generation cases. A periodic backup failure stops prework and keeps local progress; wait for the backup acknowledgment before replacing a runtime.

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib, json, os, sys
TRAINING_PROFILE = 'memory_safe'  # Later stages; Stage 1 explicitly selects 4 x 2 below
LAB_DIR = Path.cwd() / 'pacman-kev-lab'
LAB_DIR.mkdir(exist_ok=True)
COURSE_REVISION = 'bbc174c8825b8e23a20ef62856f96cd303db35d2'
FILES = {'api_client.py': 'd24f52b4bfb0c85fa1f4e67ab295eaa08e0f5654e71e6f0e0b030d7fb5347de6', 'pacman_lab.py': 'ebfa966b1a8f4bbaf90c98eacb11e77f2c18ec04158bee6744d1aee960307bc8', 'cloud_runtime.py': '9ad546991e4542d5d7d540b67d49218b0d3f6a656c209385a5111c0ff8deda52', 'training_monitor.py': '816f46da63ab7e936d55e85425ab9213145f2276fe6f2a260248307b45fbe681', 'lora_recovery.py': 'd01bc4f336b17948dc89fa8e931761c766fed979349a310d19f23e5ce0d6e274', 'checkpoint_backup.py': 'a59c305ae279170a069b8194ff176800ca9702a3651dcb92442e2631ba18db7c', 'optimized_training.py': '5c33fac17ec6ad6439193737aea650f85cdfb4445a0d346bb5bed10b858fcdfa', 'training-kernels.json': '24dff024433f61c4a571f16326f4f16c5666297e6514089e66e721b39765e56c', 'training_stages.py': '96fdd59285b8872b946523a88d95393decb73536322ca35e8f67f79cb2a7ef5c', 'training-stages.json': '9be232b730a9550a01011091b1abf3d79eaba3de8db5bdd455783fe5ae0e8ad0', 'games/arcade-engine.js': 'df5cf83c69810c641b65ccb533dd5da5b0ca6aca619db32090ad932fd8cd7e16', 'games/arcade-worker.js': 'acd16571bd330e2dc8033ab7c59977f192ae634d1eec4291db7e16a09b730daa', 'games/arcade-browser.js': 'e6f7fcba0e8d5c79573fe461e89efb428584acea594e9ec1e351e5d2964a0395', 'games/arcade-shell.html': 'b2606cdd679242c434570fd3850618041f79a8590f3b813106536c4d78341a0c', 'vendor/arcade-pacman/source.json': 'a0f21cf6009b7f7ca195fb8ce5f0730609767e7ced1c9b8cf29152f5bbe52545', 'vendor/arcade-pacman/source.zip': 'fd9ae0cbeebdfd798c609464e884f6e9c9b59673bd50083fbd0e32ea6c26a6e8', 'planner_data.py': '9b325f71bdc08419014b6e69b1541868d483c20efaaf79964892ac3887e65ca4', 'games/arcade-planner.js': 'f505a2b131465f0f170e16731863efe870e8549b50239f41d5a230dd57b7ff57', 'data/pacman-planner-v1.zip': 'a6f7e5282e888ede6a24f28cc7dffdb7d3cf51776ec5c9d6adefcf6e15e9e6a1', 'data/pacman-planner-v1-manifest.json': '3c22ede98985264b46410dc14026eaa22301b79a3f9061cf5063c739fc39b6e2', 'data/pacman-planner-v1-quality.json': 'db6ae0d8b730c7582f3f76e0c0177061d5449cfd55bc87ae3dd4c66277d54fc0', 'gameplay_benchmark.py': '62f205f9db11c7e067dbd07f609d0835fe92e1d2de3627aee2f7b641d2ed2008', 'benchmark-spec.json': '6be3f95e110422defe40c5c9007617f9e09f8c0e5c86a16b9107a73e52512cdc', 'games/benchmark-hooks.js': '9087a8ee70d8bb25c71a84abe8f614a4ea99f897ba345e36f4a70f520dca4069', 'games/benchmark-worker.cjs': '1d30685f16f5c55041fc5c50156da1119bd96c2ef382e09d4c7048939f7b1df7', 'games/arcade-teacher.js': '66d98198b77ad5364688006ed132804fae1ccc395fb4a4536980abbdafbe2f3b', 'teacher_validation.py': '0dfd7e7965a4837a7a989831802253d7f57a830283b241bf659ade7c876e8084', 'teacher-validation-spec.json': 'f5a57378e7020e543b14d4ab8ebaa0628097393307c2c493b96c0eb26d736780', 'teacher_data.py': '095a75cd23d2386987ce40d250b0fbd1979a01a97265ebc3a397aec749a847f0', 'evaluation/teacher-qualification.json': 'bd848daed094f60e3c84dc267f2569f644bf8ce04c324d79942bf0dd821dbce5', 'evaluation/teacher-qualification-replays.zip': 'bb21bcbf33ab5332c35a4f6022f8e4a1a4f70ef1546c0e5c3218142fc8d520c2', 'data/pacman-native-v2.zip': '8327bd809c86172283366cf018321e59ba91ceafc2d798555b28cb0aab6e7651', 'data/pacman-native-v2-manifest.json': '8cc0a7b26b010ff4bb1866e37f5bb3f5841c3c4a6f81d932d9ae52ba4971ae0f', 'data/pacman-native-v2-replays.zip': '6cfb6378951af88039f9503c13964a47cf83f7adaeb3d32245e69342b9652545', 'data/pacman-native-v2-generator.py': '31b7d5dcd4f750db0698f22535f13667f79e77e97cee525f373b8f99a9326be9', 'v3_data.py': '5220adf63df9326ac05c0581e7bffbb682ca1b4df4709152dd2a48e276c8d2d1', 'v4_data.py': '8a35c4f907d1391471b23ce11f6006141f931e6b00a00b408046e2a2ed0d6063', 'teacher_offline_data.py': 'e35f4ba38b45807637c8a4502326a31656c89071ee74fb2bf0a2d2f8e936ef48', 'scenario_catalog.py': '1962c676744570214a6d369ecd8f10a9343db16e039a85e2f73468c04f3570b4', 'collection_backup.py': '1f5b7b8f8412786ba504ed640720c931a44381e61b9557b457423d4bb125c625', 'hard_case_evaluation.py': '6827a4fdc52980f9c1b1240a1d1879e3cace8ea0ef8f6b23c8edc0d7a583c0b7', 'evaluation/v3-analysis-2026-10-07/teacher-reference.json': 'c3c06bb5a5de1cca6559da62206a1314aefa17c544c97bd7171b22903014c888'}
for name, expected_sha256 in FILES.items():
    target = LAB_DIR / name
    if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() == expected_sha256:
        continue
    url = f'https://raw.githubusercontent.com/yxc20089/QPlusLearning/{COURSE_REVISION}/labs/lab-01-kev-pacman/{name}'
    with urlopen(url, timeout=60) as response:
        content = response.read()
    assert hashlib.sha256(content).hexdigest() == expected_sha256, f'Unexpected source: {name}'
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(content)
    print('Downloaded verified helper:', name, flush=True)
sys.path.insert(0, str(LAB_DIR))
# Refresh helpers when upgrading an existing notebook; keep checkpoints/logs.
if 'runtime' in globals():
    runtime.stop()
import importlib
importlib.invalidate_caches()
for name in ['cloud_runtime', 'training_monitor', 'lora_recovery', 'checkpoint_backup', 'optimized_training', 'training_stages', 'gameplay_benchmark', 'teacher_validation', 'teacher_data', 'planner_data', 'pacman_lab', 'api_client', 'v3_data', 'v4_data', 'collection_backup', 'hard_case_evaluation']:
    sys.modules.pop(name, None)
from cloud_runtime import CloudRuntime
from lora_recovery import latest_snapshot
from checkpoint_backup import new_run_directory
from training_stages import specifications, inspect_checkpoint, restore_checkpoint, backup_checkpoint
from pacman_lab import *
from teacher_data import prepare_dataset, PREFIX, RECIPE
from gameplay_benchmark import benchmark_gameplay, paired_gameplay
from api_client import call, distribution
ensure_node(LAB_DIR)
runtime = CloudRuntime(LAB_DIR, training_profile=TRAINING_PROFILE)
CHECKPOINT_ROOT = LAB_DIR / 'checkpoints'
INITIAL = CHECKPOINT_ROOT / 'kev-4b-initial'
DATES = CHECKPOINT_ROOT / 'kev-4b-dates'
DOCUMENTS = CHECKPOINT_ROOT / 'kev-4b-documents'
SKILLS = CHECKPOINT_ROOT / 'kev-4b-skills'
STAGES = specifications()
manifest = None  # Loaded with its teacher qualification receipt in CP1
GAME = notebook_game(action_version=2)
STAGE_OWNERS = {}
print('Prepared the native arcade engine, four ghosts and full-game benchmark. CP1 verifies qualified demonstrations.')

In [ ]:
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'tensorboard==2.20.0'])
LOG_DIR = LAB_DIR / 'logs'
LOG_DIR.mkdir(exist_ok=True)
SHOW_TENSORBOARD = True
if SHOW_TENSORBOARD:
    from IPython import get_ipython
    ip = get_ipython()
    if ip is not None:
        ip.run_line_magic('load_ext', 'tensorboard')
        ip.run_line_magic('tensorboard', f'--logdir "{LOG_DIR}" --reload_interval 5')
    else:
        print('Outside a notebook, run: tensorboard --logdir', LOG_DIR)

In [ ]:
runtime.setup()

In [ ]:
RUN_TRAINING_PREFLIGHT = False  # Optional extra model checks; optimized bindings remain required
audit = runtime.prepare_training(run_training_preflight=RUN_TRAINING_PREFLIGHT)
print('Training suite records:', audit['suite_records'])
print('Trainable parameters:', sum(audit['trainable_parameters'].values()))
print('Optimized training verification:', json.dumps(runtime.training_preflight, indent=2))

In [ ]:
SAVE_TO_DRIVE = True  # True enables automatic backup and restore; no manual file copying
RESTORE_RECOVERY_ARCHIVE = None  # ZIP from the inspection/export cell; restore into an absent root
CHECKPOINT_ROOT = LAB_DIR / 'checkpoints'
DRIVE_BACKUP_ROOT = None
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BACKUP_ROOT = Path('/content/drive/MyDrive/QPlusLearning/lab-01-kev-pacman/backups')
if RESTORE_RECOVERY_ARCHIVE is not None:
    restore_checkpoint(RESTORE_RECOVERY_ARCHIVE, CHECKPOINT_ROOT)
else:
    CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
print('Automatic Drive backup:', json.dumps(runtime.configure_backup(DRIVE_BACKUP_ROOT, CHECKPOINT_ROOT), indent=2))
print('Later-stage training profile:', runtime.training_profile, 'overrides:', runtime.selected_recipe({}))
print('Stage 1 selects batch 4 x accumulation 2 / row_budget 0 in its own training cell.')
print('Checkpoints and recovery:', CHECKPOINT_ROOT)

## Freeze the completed v3 learner

The parent is `checkpoints/kev-4b-pacman-native-v3`, restored from its own Drive backup. Check the printed model hash and rank. Training/development use new disjoint seed families; the original 20 starts remain evaluation-only. Changing parent weights or collection configuration requires a separate collection directory.

In [ ]:
from v4_data import (harvest_scenarios, collect_learner, probe_candidates, verify_cases,
                     build_dataset, validate_dataset, completed_files, PREFIX as V4_PREFIX,
                     RECIPE as V4_RECIPE, DEFAULT_TRAIN_SEEDS as V4_TRAIN_SEEDS,
                     DEFAULT_DEVELOPMENT_SEEDS as V4_DEVELOPMENT_SEEDS)
from collection_backup import restore_collection, backup_collection
from hard_case_evaluation import evaluate_hard_cases, native_summary
from training_stages import checkpoint_fingerprint
from api_client import call_batch
V3 = CHECKPOINT_ROOT / 'kev-4b-pacman-native-v3'
V4 = CHECKPOINT_ROOT / 'kev-4b-pacman-native-v4'
V4_DATA = LAB_DIR / 'data'
V4_TRAINING = V4_DATA / f'{V4_PREFIX}-train.jsonl'
V4_DEVELOPMENT = V4_DATA / f'{V4_PREFIX}-development.jsonl'
QUALIFICATION = LAB_DIR / 'evaluation/teacher-qualification.json'
COLLECTION_DIR = LAB_DIR / 'results/v4-collection'
COLLECTION_WAVE = 'wave-001'
WAVE_TRAIN_SEEDS = list(V4_TRAIN_SEEDS)
WAVE_DEVELOPMENT_SEEDS = list(V4_DEVELOPMENT_SEEDS)
OFFLINE_ROOTS_PER_EPISODE = 64
LEARNER_ROOTS_PER_EPISODE = 64
COLLECTION_BACKUP = (runtime.backup_root.parent / 'prework-backups/v4-collection'
                     if runtime.backup_root is not None else None)
if COLLECTION_BACKUP is not None:
    restore_collection(COLLECTION_DIR, COLLECTION_BACKUP)
if not (V3 / 'run-evidence.json').is_file():
    raise RuntimeError(f'Restore the completed native-v3 adapter to {V3}. This round does not start from Skills or v2.')
runtime.start(V3)
V3_IDENTITY = runtime.active_model_info()
assert V3_IDENTITY['lora_rank'] == 16 and V3_IDENTITY['pacman_fine_tuned']
def predict_v3_batch(requests):
    if runtime.active_model_info() != V3_IDENTITY:
        raise RuntimeError('Start the frozen v3 checkpoint before probing candidates')
    responses = [value for value, elapsed in call_batch('/v1/systemone', requests, concurrency=16)]
    if runtime.active_model_info() != V3_IDENTITY:
        raise RuntimeError('Adapter changed during candidate probing')
    return responses
def generation_needed():
    return not ((V4_DATA / f'{V4_PREFIX}-manifest.json').is_file() or (V4 / 'run-evidence.json').is_file())
print('Frozen learner / warm-start parent:', json.dumps(V3_IDENTITY, indent=2))
print('New v4 output:', V4)
print('Collection Drive backup:', COLLECTION_BACKUP)
print('Selected recipe:', V4_RECIPE)

## 1. Harvest targeted offline native scenarios — CPU prework

New native games provide coherent board state, movement history and clocks. Teacher prefixes and legal deviations expose impending ghost interceptions, late retreat, power expiration, pellet stalls, sparse-food cleanup and respawn. No hand-edited impossible board or future RNG enters Kev's input. This creates **candidates**, not accepted labels. Completed per-episode receipts are reused. Automatic workers respect physical cores, affinity and CPU quota; set a positive integer to override.

Generation backs up completed artifacts and changed indexes at start, every five minutes and orderly stop. Active/incomplete work retries. A hard runtime loss can lose work since the last acknowledged backup. Do not run this cell concurrently with another generation stage in the same collection.

The default is a first collection wave, not a promise of enough qualifying cases. If admission reports shortages, set a new `COLLECTION_WAVE` and new disjoint `WAVE_TRAIN_SEEDS`/`WAVE_DEVELOPMENT_SEEDS` in the loading cell, rerun it, then stages 1–4. Prior completed cases remain reusable. A saved wave cannot change its seeds or limits. Never use the 500xx benchmark seeds.

In [ ]:
TEACHER_WORKERS = None
if generation_needed():
    runtime.stop()
    offline_status = harvest_scenarios(COLLECTION_DIR, QUALIFICATION,
        train_seeds=WAVE_TRAIN_SEEDS, development_seeds=WAVE_DEVELOPMENT_SEEDS,
        wave_id=COLLECTION_WAVE, roots_per_episode=OFFLINE_ROOTS_PER_EPISODE,
        workers=TEACHER_WORKERS, backup_directory=COLLECTION_BACKUP)
    print(json.dumps(offline_status, indent=2))
else:
    print('Verified dataset/completed v4 restored; generation skipped.')

## 2. Collect fresh v3 trajectories — GPU prework

The learner plays fresh training/development games without teacher overrides. Store complete state/action/probability traces and adapter identity. These states complement offline targeted scenarios. Entire source episodes and their forks stay in the same partition; benchmark seeds cannot be used. Completed games are reused when rerunning with identical weights/configuration.

In [ ]:
if generation_needed():
    runtime.start(V3)
    learner_status = collect_learner(COLLECTION_DIR, model_info=runtime.active_model_info, qualification_path=QUALIFICATION,
        train_seeds=WAVE_TRAIN_SEEDS, development_seeds=WAVE_DEVELOPMENT_SEEDS,
        wave_id=COLLECTION_WAVE, roots_per_episode=LEARNER_ROOTS_PER_EPISODE,
        predict=lambda request: predict_v3_batch([request])[0],
        backup_directory=COLLECTION_BACKUP)
    print(json.dumps(learner_status, indent=2))
else:
    print('Verified dataset/completed v4 restored; learner collection skipped.')

## 3. Probe candidate states with the frozen v3 model — GPU

Send independent states concurrently to Kev's existing HTTP batching queue. Keep each legal wrong alternative in the request; gold labels, teacher plans and future outcomes stay outside the input. Save v3's move and raw probabilities. Confident wrong choices are priority corrections; agreement on a dangerous-looking state is not a verified disagreement. Rerun after verification if it produces a bounded informative candidate window.

In [ ]:
if generation_needed():
    runtime.start(V3)
    probe_status = probe_candidates(COLLECTION_DIR, model_info=runtime.active_model_info,
        predict_batch=predict_v3_batch, batch_size=16, backup_directory=COLLECTION_BACKUP)
    print(json.dumps(probe_status, indent=2))
else:
    print('Verified dataset/completed v4 restored; probing skipped.')

## 4. Verify consequential disagreements — CPU

The positive teacher branch must clear the native maze and pass the existing survival/sustained-loop/stall gates. Immediate fatal alternatives use exact native one-action counterfactuals. Earlier ghost traps, stalls and detours compare a forced teacher-first move with a forced v3-first move followed by the **same frozen teacher**. A different successful safe route alone does not qualify as a harmful negative.

Retain full successful and rejected continuations. Independently replay admitted cases. Short informative lead-in/recovery windows need their own provenance and probe; the returned `pending_probe_count` tells you whether to rerun cells 3 then 4. Completed verification is reused. Do not train until all selected evidence and quotas pass.

In [ ]:
if generation_needed():
    runtime.stop()
    verification_status = verify_cases(COLLECTION_DIR, QUALIFICATION,
        workers=TEACHER_WORKERS, backup_directory=COLLECTION_BACKUP)
    print(json.dumps(verification_status, indent=2))
    if verification_status.get('pending_probe_count', 0):
        print('Informative candidates remain: rerun probe cell 3, then verification cell 4.')
else:
    print('Verified dataset/completed v4 restored; verification skipped.')

## 5. Admit the hard-focused dataset

Target **3,660 verified consequential disagreement roots + 1,220 informative hard decisions + 912 prior Pac-Man replay**, with **304 generic decision requests** added by Kev's trainer. The Pac-Man file has 5,792 requests; total training stays 6,096 / 762 updates. At least 75% of the 4,880-case hard pool must be verified disagreement roots. Deduplication, source/cohort quotas and episode caps prevent correlated easy frames from dominating.

Fresh development remains separate. Insufficient distinct hard cases stop this cell and report missing quotas. Expand targeted collection on new seeds in a separate collection wave; do not duplicate rows, relabel benchmark traces or enable partial-data training. Full trajectories are evidence, not automatically all training rows. The selected states use native history already included in the observation.

In [ ]:
if not (V4 / 'run-evidence.json').is_file():
    v4_manifest = build_dataset(COLLECTION_DIR, V4_DATA, QUALIFICATION)
    assert v4_manifest['probe_checkpoint']['checkpoint_sha256'] == checkpoint_fingerprint(V3)
    assert v4_manifest['expected_training_requests'] == 6096
    assert v4_manifest['expected_optimizer_steps'] == 762
    print(json.dumps({k: v4_manifest[k] for k in
        ['counts', 'selected_budgets', 'coverage', 'expected_optimizer_steps']}, indent=2))
    if COLLECTION_BACKUP is not None:
        backup_collection(COLLECTION_DIR, COLLECTION_BACKUP, files=completed_files(COLLECTION_DIR))
else:
    print('Completed v4 restored; dataset build skipped.')

## 6. Train a new v4 stage from v3 weights

Rank 16, all-module LoRA/head, existing CE, one epoch, lr 2e-5, batch 4 × accumulation 2, required optimized kernels, BF16 autocast and FP32 stored weights. Changing data selection is the first experiment; a pairwise loss or higher rank is a later ablation. Rerunning resumes only an interrupted **v4** optimizer. Drive checkpoint backup also saves the exact v4 input and its development/verification receipt. Watch recovery/backup acknowledgments at step 1, every 100 steps or five minutes, and completion.

In [ ]:
if (V4 / 'run-evidence.json').is_file():
    print('Using completed v4 checkpoint:', V4)
else:
    v4_manifest = validate_dataset(V4_DATA, QUALIFICATION)
    assert v4_manifest['probe_checkpoint']['checkpoint_sha256'] == checkpoint_fingerprint(V3)
    assert v4_manifest['expected_training_requests'] == 6096
    assert v4_manifest['expected_optimizer_steps'] == 762
    assert v4_manifest['recipe'] == V4_RECIPE
    RESUME_V4 = latest_snapshot(Path(str(V4) + '-recovery')) is not None
    print('Resume interrupted v4:', RESUME_V4)
    print('Selected v4 command:', runtime.finetuning_command(V4_TRAINING, V4, V3, recipe=V4_RECIPE))
    runtime.finetune(V4_TRAINING, V4, init_from=V3, steps=0, resume=RESUME_V4, recipe=V4_RECIPE)
    metrics = json.loads((V4 / 'training_metrics.json').read_text())
    assert metrics['optimizer_steps'] == 762 and metrics['records_seen'] == metrics['requested_records'] == 6096
    assert not metrics.get('truncated_records', 0) and not metrics.get('rejected_records', 0)
    evidence = json.loads((V4 / 'run-evidence.json').read_text())
    assert evidence['parent_checkpoint_sha256'] == checkpoint_fingerprint(V3)
print('V4 training metrics:', (V4 / 'training_metrics.json').read_text())

## Compare v3/v4 on frozen hard development and full native games

Both models receive the same disjoint hard development labels for diagnostic teacher agreement, CE and teacher probability margin by cohort. Snapshot fit remains secondary. The primary comparison uses the **unchanged 20 native game starts**, continuing through lives until clear, game-over or watchdog. No teacher or safety filter replaces a model choice. Save all trajectories; report clears/censoring, first-life food, pellet/native-frame efficiency, avoidable deaths and their opportunity denominators, all-actions-fatal endpoints, dry intervals and sustained cycles. A short safe retreat can be correct even when it collects no immediate pellet.

This cell also retains the archived, independently replay-verified frozen teacher reference on these same 20 starts: 20 clears, one life lost. The reference is checked against the current protocol, teacher sources/options and qualification hash; it is context from the prior audit, not a new teacher run. Its private raw traces were retained in the instructor archive. This cell creates a new dated recording directory each time. Pause interactive play first. Tune using fresh development games; these 500xx benchmark traces remain evaluation-only.

In [ ]:
comparison_run = new_run_directory(LAB_DIR / 'results', 'comparison-v3-v4',
    baseline_checkpoint=str(V3), fine_tuned_checkpoint=str(V4), action_version=2)
validate_dataset(V4_DATA, QUALIFICATION)
development = {}
gameplay = {}
for name, checkpoint in [('native-v3', V3), ('native-v4', V4)]:
    runtime.start(checkpoint)
    development[name] = evaluate_hard_cases(V4_DEVELOPMENT,
        predict_batch=lambda requests: [value for value, elapsed in call_batch('/v1/systemone', requests, concurrency=16)],
        model_info=runtime.active_model_info)
    (comparison_run / f'{name}-hard-development.json').write_text(json.dumps(development[name], indent=2))
    gameplay[name] = benchmark_gameplay(model_info=runtime.active_model_info, trace_dir=comparison_run / name)
    (comparison_run / f'{name}.json').write_text(json.dumps(gameplay[name], indent=2))
comparison = {'schema_version': 4, 'baseline_checkpoint': str(V3), 'fine_tuned_checkpoint': str(V4),
    'hard_development': development,
    'gameplay': {'general': gameplay['native-v3'], 'fine_tuned': gameplay['native-v4'],
                 'paired': paired_gameplay(gameplay['native-v3'], gameplay['native-v4'])},
    'behavior': {name: native_summary(report) for name, report in gameplay.items()},
    'recording_directory': str(comparison_run)}
from teacher_validation import source_hashes
teacher_reference = json.loads((LAB_DIR / 'evaluation/v3-analysis-2026-10-07/teacher-reference.json').read_text())
assert teacher_reference['simulation_protocol'] == gameplay['native-v3']['protocol']
assert teacher_reference['source_sha256'] == source_hashes()
assert teacher_reference['qualification_sha256'] == hashlib.sha256(QUALIFICATION.read_bytes()).hexdigest()
assert teacher_reference['options'] == json.loads(QUALIFICATION.read_text())['options']
assert teacher_reference['native_replays_verified']
comparison['teacher_reference'] = {'summary': teacher_reference['summary'],
    'source_sha256': teacher_reference['source_sha256'],
    'teacher_replay_bundle_sha256': teacher_reference['teacher_replay_bundle_sha256'],
    'interpretation': 'Archived frozen-teacher games on the same20 starts, independently replay-verified in the prior audit; not regenerated by this cell.'}
(comparison_run / 'comparison.json').write_text(json.dumps(comparison, indent=2))
(LAB_DIR / 'comparison-v3-v4.json').write_text(json.dumps(comparison, indent=2))
print('Native safety and food efficiency:', json.dumps(comparison['behavior'], indent=2))
print('All development probes and benchmark trajectories:', comparison_run)

## Interactive play with native-v4 — ungraded

Choose Kev and Start. The badge must say `kev-4b-pacman-native-v4` and show its LoRA/head hashes. The observation/action protocol and native ghost/pellet mechanisms match the benchmark. Every invocation saves a separate dated trace. Pause before evaluation or backup.

In [ ]:
# Interactive play with the completed Pac-Man adapter
from IPython.display import display, HTML, JSON
PACMAN_PLAY_CHECKPOINT = CHECKPOINT_ROOT / 'kev-4b-pacman-native-v4'
# To inspect your existing v1 adapter, explicitly select CHECKPOINT_ROOT / 'kev-4b-pacman-planner-v1' instead.
if not (PACMAN_PLAY_CHECKPOINT / 'run-evidence.json').is_file():
    raise RuntimeError(f'Complete Stage 5 or restore its completed checkpoint to {PACMAN_PLAY_CHECKPOINT} before playing.')
if runtime.active_checkpoint is None or runtime.active_model_info()['checkpoint'] != str(PACMAN_PLAY_CHECKPOINT):
    runtime.start(PACMAN_PLAY_CHECKPOINT)
if not runtime.active_model_info()['pacman_fine_tuned']:
    raise RuntimeError('The active checkpoint is not recorded as Pac-Man fine-tuned. Load the completed Stage 5 checkpoint.')
fine_tuned_play_run = new_run_directory(LAB_DIR / 'results', 'interactive-fine-tuned',
                                       active_checkpoint=runtime.active_model_info(), seed=7, action_version=2)
fine_tuned_bridge = NotebookBridge(fine_tuned_play_run / 'fine-tuned-player-trace.jsonl', runtime.active_model_info)
print('Interactive state/action/probability trace:', fine_tuned_bridge.trace_path)
try:
    from google.colab import output
except ImportError:
    print('Kaggle/local: use the Python rollouts below; interactive callbacks require Colab.')
else:
    output.register_callback('pacman.decide', lambda state, selected_bridge=fine_tuned_bridge: JSON(selected_bridge.decide(state)))
    output.register_callback('pacman.model', lambda selected_bridge=fine_tuned_bridge: JSON(selected_bridge.model()))
    display(HTML(GAME))

## Final Drive backup: checkpoints, cases, curves and full traces

Run before replacing the runtime. The session archive saves all generation evidence, accepted/rejected branches, benchmark trajectories, hard-development probes and interactive traces. Checkpoint backups remain separate for each adapter and include v4's exact input/development/receipt. Wait for `Session backup complete` and an empty `checkpoint_errors` list.

In [ ]:
# Back up all available checkpoints, trajectories and traces to Google Drive
from pathlib import Path
import json
from google.colab import drive
from checkpoint_backup import backup_lab_to_drive
drive.mount('/content/drive')
runtime.stop()
DRIVE_LAB_ROOT = Path('/content/drive/MyDrive/QPlusLearning/lab-01-kev-pacman')
drive_receipt = backup_lab_to_drive(LAB_DIR, DRIVE_LAB_ROOT, checkpoint_root=CHECKPOINT_ROOT)
print(json.dumps(drive_receipt, indent=2))
if drive_receipt['checkpoint_errors']:
    raise RuntimeError('Traces were saved, but some checkpoint backups failed. See checkpoint_errors above and retry.')